[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/beltrami_fields/beltrami_fields.ipynb)

# Beltrami Fields

This example computes Beltrami fields, the force-free states of a magnetic field, in which the electric current follows the field, so that the field exerts no force on the current that carries it. Magnetic fields keep their linkage and twist even as their finer structure fades, and the force-free state of least energy is where a field of given linkage settles.

The field is static, so the example lives in one instant of spacetime, `x+y+z+t-`, with the direction of time `time` of an observer at rest with the torus. The magnetic field is a bivector, a plane of space, and an oriented patch reads its flux as `field & patch`. Positions and patches are directions and patches of space; a hat function is constant in time, so its gradient is the volume its opposite face sweeps through time, read against the prism the tetrahedron sweeps, `volume & time`. Whitney reconstruction connects those local readings to mesh edges; its adjugate brings energy and signed-volume helicity back to the same edge amplitudes.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np
from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from examples.electromagnetism.beltrami_fields import render, scenarios
from examples.electromagnetism.beltrami_fields.plumbing import (
    as_scalar, flux_preserving, solid_torus, edge_tails, edge_heads, opposite_tails, opposite_heads,
    face_first, face_second, face_third,
)
from examples.electromagnetism.beltrami_fields.numerical import coexact_helicity_eigh
from examples.util.animation import save_figure, show_animation

ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()          # Scalar+
Plane = ga.gatype.vector()           # Vector+
# The magnetic field, a plane of space.
Field = ga.gatype.bivector().derive.within("x y z")  # Bivector+
Point = ga.gatype.antivector()        # Antivector-
Curl = ga.gatype((Field, Scalar))

# The direction in time of an observer at rest with the torus, which pairs to one with `mv.t`.
time = -mv.t.dual()                  # [] Point-

## Forms on a tetrahedron

Each edge form reads one along its own edge and zero along the others. Its circulation turns around the opposite edge; a face form reads flux through its own face and vanishes on the other faces. The signed-volume pairing between edge forms and curls survives stretching and shearing of the tetrahedron.

In [ ]:
n_tet_subdivisions = 6

In [ ]:
reference_corners, samples = scenarios.single_tetrahedron(n_tet_subdivisions)  # [corners] Point-, [samples] Point-
reference_volume = ((reference_corners[1] - reference_corners[0]) & (reference_corners[2] - reference_corners[0]) & (reference_corners[3] - reference_corners[0])) / 6  # [] Volume-
# The prism the tetrahedron sweeps in a unit of time; its size is the tetrahedron's volume.
reference_prism = reference_volume & time                                      # [] Scalar-
reference_faces = ((reference_corners[face_second] - reference_corners[face_first]) & (reference_corners[face_third] - reference_corners[face_first])) / 2  # [corners] Patch-
# Each hat is constant in time: the volume its opposite face sweeps through time, over the prism,
# gives the plane reading a corner hat's rise.
hat_gradients = (reference_faces & time) / (3 * reference_prism)               # [corners] Plane+
hats = 1 + (hat_gradients[:, None] & (samples[None, :] - reference_corners[:, None]))  # [corners, samples] Scalar+

# Joining the opposite edge with a displacement gives the patch read by the edge form.
opposite_edges = reference_corners[opposite_heads] - reference_corners[opposite_tails]  # [tet_edges] Point-
swept_patches = (samples[None, :] - reference_corners[opposite_tails, None]) & opposite_edges[:, None]  # [tet_edges, samples] Patch-
edge_forms = (swept_patches & time) / (6 * reference_prism)                    # [tet_edges, samples] Plane+
edge_curls = (opposite_edges & time) / (3 * reference_prism)                   # [tet_edges] Field+
face_forms = ((samples[None, :] - reference_corners[:, None]) & time) / (3 * reference_prism)  # [tet_faces, samples] Field+

# A linear edge form and constant curl integrate exactly at the centroid.
centroid_forms = (hat_gradients[edge_heads] - hat_gradients[edge_tails]) / 4   # [tet_edges] Plane+
pair_densities = ((centroid_forms[:, None] ^ edge_curls[None, :]) + (centroid_forms[None, :] ^ edge_curls[:, None])) / 2                # [tet_edges, tet_edges] Trivector+
tet_helicity = pair_densities & reference_volume                             # [tet_edges, tet_edges] Scalar+

save_figure(render.draw_tet_whitney(reference_corners, samples, edge_forms, face_forms, reference_volume), "beltrami_tet_whitney");

An edge form circulates around the opposite edge, a face form radiates from the opposite corner, and only opposite edges link in the helicity pairing. On the left an edge is orange and its opposite edge blue; in the middle the face is orange; on the right the rings circulate around the opposite edge.

In Whitney-form notation, the same edge and face fields read as $w_{ij}=\lambda_i d\lambda_j-\lambda_j d\lambda_i$ and $w_{ijk}=2(\lambda_i d\lambda_j\wedge d\lambda_k+\lambda_j d\lambda_k\wedge d\lambda_i+\lambda_k d\lambda_i\wedge d\lambda_j)$.

## Beltrami modes in a torus

Energy reads the field through the vacuum, while helicity reads its linkage through signed volume. Both are forms on the edge amplitudes, and a Beltrami mode makes their responses proportional. A reflected field, read on the same positively oriented tetrahedra, keeps its energy and reverses its helicity, so the modes come in pairs of opposite handedness.

In [ ]:
major_radius = 1.0
minor_radius = 0.42
n_major = 16
n_minor = 8
shown_indices = np.array([-1, -4])

In [ ]:
mesh, vertices = solid_torus(major_radius, minor_radius, n_major, n_minor)
corners = mesh.incidence[3, 0] * vertices                                  # [corners] Point-[tets]
centres = corners.sum(axis=0) / 4                                          # Point-[tets]
volumes = ((corners[1] - corners[0]) & (corners[2] - corners[0]) & (corners[3] - corners[0])) / 6  # Volume-[tets]
prisms = volumes & time                                                    # Scalar-[tets]
face_patches = ((corners[face_second] - corners[face_first]) & (corners[face_third] - corners[face_first])) / 2  # [corners] Patch-[tets]

# The corner planes read each hat's rise across its opposite face.
corner_gradients = (face_patches & time) / (3 * prisms)                    # [corners] Plane+[tets]

# At each tetrahedron centroid the four barycentric hats equal 1/4, so the edge 1-form
# reduces to one quarter of the difference of its endpoints' corner gradients.
centroid_1forms = (corner_gradients[edge_heads] - corner_gradients[edge_tails]) / 4  # [tet_edges] Plane+[tets]
# A tet's edges run from its lower local corner to its higher, the mesh's edges from the lower vertex to the
# higher: `mesh.oriented[3, 1]` reads each mesh edge with the sign of their agreement.
whitney1 = (spdiag(centroid_1forms * Scalar) * mesh.oriented[3, 1]).sum(axis=0)  # (Plane+ <- Scalar+)[tets <- edges]

# The wedge of endpoint gradients reads the circulation through a patch.
curls = 2 * (corner_gradients[edge_tails] ^ corner_gradients[edge_heads])  # [tet_edges] Field+[tets]
curl = (spdiag(curls * Scalar) * mesh.oriented[3, 1]).sum(axis=0)          # (Field+ <- Scalar+)[tets <- edges]

In [ ]:
# Magnetic energy pulls the vacuum, which takes each tetrahedron's field to the excitation it causes
# weighted by its volume, back through the curl; magnetic helicity pulls the metric-free volume
# pairing back through the centroid 1-form and the curl.
vacuum = spdiag(prisms.norm() * Field.dual())                              # (Patch- <- Field+)[tets <- tets]
stiffness = curl.adjugate()(vacuum(curl))                                  # (Amount- <- Scalar+)[edges <- edges]
volume_weights = spdiag(Plane ^ volumes)                                   # (Patch- <- Plane+)[tets <- tets]
raw_helicity = curl.adjugate()(volume_weights(whitney1))                   # (Amount- <- Scalar+)[edges <- edges]
helicity = (raw_helicity + raw_helicity.adjugate()) / 2                    # (Amount- <- Scalar+)[edges <- edges]

The conducting wall carries no outward flux. A circulation around the tube remains free, allowing magnetic flux around the torus.

In [ ]:
free = flux_preserving(mesh, n_major, n_minor)                             # (Scalar+ <- Scalar+)[edges <- free]
# Directions with zero magnetic energy change the potential without changing its field.
free_energy = free.adjugate()(stiffness(free))                             # (Amount- <- Scalar+)[free <- free]
free_helicity = free.adjugate()(helicity(free))                            # (Amount- <- Scalar+)[free <- free]
inverse_eigenvalues, free_modes, _ = coexact_helicity_eigh(free_energy, free_helicity)
modes = free(free_modes)                                                   # [modes] Scalar+[edges]
shown_modes = modes[shown_indices]                                         # [shown] Scalar+[edges]
shown_eigenvalues = 1 / inverse_eigenvalues[shown_indices]                 # [shown] Scalar+
shown_fields = curl(shown_modes)                                           # [shown] Field+[tets]

save_figure(
    render.draw_beltrami_eigenmodes(mesh, vertices, centres, volumes, inverse_eigenvalues, shown_fields, shown_indices),
    "beltrami_eigenmodes",
);

The spectrum of inverse eigenvalues, each mode's helicity per unit squared energy norm, pairs positive and negative helicities. The two marked modes appear beside it: the lowest positive-helicity mode and a higher mode with more internal structure. Tubes follow a smoothed, wall-confined reconstruction of the tetrahedral field; colours distinguish seed layers, and arrowheads indicate direction.

In the matrix notation of finite elements, the energy and helicity forms give $Sa=\lambda Ha$, with $S_{ee'}=\int dw_e\wedge\star dw_{e'}$ and $H_{ee'}=\frac12\int(w_e\wedge dw_{e'}+w_{e'}\wedge dw_e)$.

## Selective decay

Fine twists fade faster than broad circulation. The amplitudes below follow a prescribed damping law with a shared rescaling that holds total helicity fixed, illustrating the approach toward the lowest-energy mode carrying that linkage. The two views show the same field settling into a coherent helical structure.

This cell takes about ten seconds, mainly to draw the tubes.

In [ ]:
selected_indices = np.array([-1, -2, -4, -8, 3])
initial_weights = np.array([0.45, 0.70, 0.85, 0.85, 0.45])
n_frames = 46
relaxation_time = 0.025
frame_duration_ms = 40

In [ ]:
def selective_decay(initial: Scalar, eigenvalues: Scalar, times: np.ndarray) -> Iterator[Scalar]:
    """Yield selectively damped modal amplitudes rescaled to preserve their initial helicity."""
    initial_helicity = (initial.squared() / eigenvalues).sum(axis=0)          # [] Scalar+
    for time in times:
        # Finer modes decay faster; the shared scale fixes the linkage of the whole field.
        damped = initial * (-eigenvalues.squared() * time).exp()              # [selected] Scalar+
        helicity = (damped.squared() / eigenvalues).sum(axis=0)               # [] Scalar+
        yield damped * (initial_helicity / helicity).square_root()


def reconstruct(states: Iterable[Scalar], modes: Scalar, curl: Curl) -> Iterator[Field]:
    """Yield the reconstructed magnetic flux field for each modal state."""
    for amplitudes in states:
        cochain = (amplitudes * modes).sum(axis=0)                            # Scalar+[edges]
        yield curl(cochain)


selected_modes = modes[selected_indices]                                    # [selected] Scalar+[edges]
selected_eigenvalues = 1 / inverse_eigenvalues[selected_indices]            # [selected] Scalar+
initial_amplitudes = as_scalar(initial_weights)                             # [selected] Scalar+
times = np.linspace(0.0, relaxation_time, n_frames)
states = selective_decay(initial_amplitudes, selected_eigenvalues, times)
fields = reconstruct(states, selected_modes, curl)
show_animation(
    render.animate_relaxation(mesh, vertices, centres, volumes, fields),
    "beltrami_relaxation", frame_duration_ms, n_frames,
)

With the helicity held fixed, the energy falls toward the least that the helicity allows, that of the Taylor state, the lowest mode alone.

In modal notation, $E=\frac12\sum_k c_k^2$ and $H=\sum_k c_k^2/\lambda_k$. Multiplying $c_k(0)e^{-\lambda_k^2t}$ by a common factor fixes $H$; for this mode mixture, the energy approaches $\lambda_1 H/2$.

In [ ]:
# checks
assert (hats.sum(axis=0) - 1).norm().sum() < 1e-13
assert hat_gradients.sum(axis=0).norm() < 1e-13
assert reference_faces.sum(axis=0).squared().square_root() < 1e-13

alternating_edges = hats[edge_tails] * hat_gradients[edge_heads, None] - hats[edge_heads] * hat_gradients[edge_tails, None]
first_hats, second_hats, third_hats = hats[face_first], hats[face_second], hats[face_third]
first_gradients, second_gradients, third_gradients = hat_gradients[face_first, None], hat_gradients[face_second, None], hat_gradients[face_third, None]
alternating_faces = 2 * (first_hats * (second_gradients ^ third_gradients) + second_hats * (third_gradients ^ first_gradients) + third_hats * (first_gradients ^ second_gradients))
assert (edge_forms - alternating_edges).norm().sum() < 1e-12
assert (face_forms - alternating_faces).norm().sum() < 1e-12

# A different tetrahedron leaves the volume-paired edge weights unchanged.
deformed_corners = scenarios.deformed_tetrahedron()
deformed_volume = ((deformed_corners[1] - deformed_corners[0]) & (deformed_corners[2] - deformed_corners[0]) & (deformed_corners[3] - deformed_corners[0])) / 6
deformed_faces = ((deformed_corners[face_second] - deformed_corners[face_first]) & (deformed_corners[face_third] - deformed_corners[face_first])) / 2
deformed_gradients = (deformed_faces & time) / (3 * (deformed_volume & time))
deformed_forms = (deformed_gradients[edge_heads] - deformed_gradients[edge_tails]) / 4
deformed_curls = 2 * (deformed_gradients[edge_tails] ^ deformed_gradients[edge_heads])
deformed_density = ((deformed_forms[:, None] ^ deformed_curls[None, :]) + (deformed_forms[None, :] ^ deformed_curls[:, None])) / 2
deformed_helicity = deformed_density & deformed_volume
assert (tet_helicity - deformed_helicity).norm().sum() < 1e-13

assert (inverse_eigenvalues + inverse_eigenvalues[::-1]).norm().sum() < 1e-11
shown_energies = (stiffness(shown_modes) & shown_modes).sites.sum() / 2
shown_helicities = (helicity(shown_modes) & shown_modes).sites.sum()
shown_residuals = free.adjugate()(stiffness(shown_modes) - shown_eigenvalues * helicity(shown_modes))
assert (shown_energies - 0.5).norm().sum() < 1e-11
assert (shown_helicities - inverse_eigenvalues[shown_indices]).norm().sum() < 1e-11
assert shown_residuals.dual_inverse().norm().sites.sum().sum() < 1e-10

amplitude_frames = stack(list(selective_decay(initial_amplitudes, selected_eigenvalues, times)))
energy_frames = amplitude_frames.squared().sum(axis=1) / 2
helicity_frames = (amplitude_frames.squared() / selected_eigenvalues).sum(axis=1)
taylor_bound = helicity_frames[0] * selected_eigenvalues[0] / 2
assert (helicity_frames - helicity_frames[0]).norm().sum() < 1e-11
assert energy_frames[-1] > taylor_bound
assert (energy_frames[-1] - taylor_bound) / taylor_bound < 0.01